# Sentiment140 markup and custom tags

Goal: apply trained classifier to big big dataset -> add score and binarylabel of toxicity/ Then create analytics

## Imports and Settings

In [1]:
!pip install pyspark findspark -q

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("ApplyToxicityModel") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 12:08:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, udf, when, lit, count, mean, stddev, expr
from pyspark.sql.functions import count, mean, sum as spark_sum, col, when
from pyspark.sql.types import FloatType
import json
from pyspark.sql.functions import desc

## Data & Model

In [4]:
SENTIMENT_PATH = "/kaggle/input/notebooks/ksksksksksksksushka/bigdata-eda/sentiment140_cleaned.parquet"
MODEL_PATH = "/kaggle/input/notebooks/ksksksksksksksushka/toxicity-classifier/toxicity_model"
METRICS_PATH = "/kaggle/input/notebooks/ksksksksksksksushka/toxicity-classifier/metrics.json"

df_sent = spark.read.parquet(SENTIMENT_PATH)
print("Sentiment140 loaded, schema:")
df_sent.printSchema()
print(f"Number of rows: {df_sent.count()}")
df_sent.show(3, truncate=50)

Sentiment140 loaded, schema:
root
 |-- user: string (nullable = true)
 |-- cleaned_text: string (nullable = true)
 |-- polarity: integer (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- hour: integer (nullable = true)

Number of rows: 1594407


+-----------+--------------------------------------------------+--------+-------+----+
|       user|                                      cleaned_text|polarity|weekday|hour|
+-----------+--------------------------------------------------+--------+-------+----+
|       TLeC|i couldnt bear to watch it and i thought the ua...|       0|      0|  22|
|KatieAngell|just going to cry myself to sleep after watchin...|       0|      0|  22|
|  labrt2004|just checked my user timeline on my blackberry ...|       0|      0|  22|
+-----------+--------------------------------------------------+--------+-------+----+
only showing top 3 rows


In [5]:
model = PipelineModel.load(MODEL_PATH)

with open(METRICS_PATH, "r") as f:
    metrics = json.load(f)

print("Model type:", metrics.get("model", "not specified"))
print("Full metrics:", metrics)

best_threshold = metrics["threshold"]
print(f"Model loaded. Best threshold: {best_threshold}")

Model type: NaiveBayes_CountVectorizer
Full metrics: {'model': 'NaiveBayes_CountVectorizer', 'threshold': 0.55, 'accuracy_weighted': 0.910279053338043, 'precision_toxic': 0.3935, 'recall_toxic': 0.6948, 'f1_toxic': 0.5025}
Model loaded. Best threshold: 0.55


## Use model!

In [6]:
predictions = model.transform(df_sent)

In [7]:
get_prob = udf(lambda v: float(v[1]), FloatType())

In [8]:
df_scored = predictions \
    .withColumn("toxicity_score", get_prob(col("probability"))) \
    .withColumn("is_toxic", when(col("toxicity_score") > best_threshold, 1).otherwise(0))

# check result
df_scored.select("user", "cleaned_text", "polarity", "toxicity_score", "is_toxic") \
    .show(5, truncate=50)

+-----------+--------------------------------------------------+--------+--------------+--------+
|       user|                                      cleaned_text|polarity|toxicity_score|is_toxic|
+-----------+--------------------------------------------------+--------+--------------+--------+
|       TLeC|i couldnt bear to watch it and i thought the ua...|       0|       1.1E-44|       0|
|KatieAngell|just going to cry myself to sleep after watchin...|       0|  3.799649E-39|       0|
|  labrt2004|just checked my user timeline on my blackberry ...|       0| 1.7925652E-34|       0|
| a_mariepyt|                      damm back to school tomorrow|       0| 3.2546375E-13|       0|
|    michrod|oh haha dude i dont really look at em unless so...|       0|    0.06865948|       0|
+-----------+--------------------------------------------------+--------+--------------+--------+
only showing top 5 rows


In [9]:
print("=== 10 random TOXIC tweets ===")
df_scored.filter(col("is_toxic") == 1) \
    .select("cleaned_text", "toxicity_score", "polarity") \
    .sample(False, 0.1, seed=42) \
    .show(10, truncate=False)

print("=== 10 random NON-TOXIC tweets ===")
df_scored.filter(col("is_toxic") == 0) \
    .select("cleaned_text", "toxicity_score", "polarity") \
    .sample(False, 0.01, seed=42) \
    .show(10, truncate=False)

=== 10 random TOXIC tweets ===


+-----------------------------------------------------------------------------------------------------------------------------------+--------------+--------+
|cleaned_text                                                                                                                       |toxicity_score|polarity|
+-----------------------------------------------------------------------------------------------------------------------------------+--------------+--------+
|no i lost a loyal                                                                                                                  |0.9990199     |0       |
|please watch this vid and tell me if you are not moved                                                                             |0.99667156    |0       |
|i really dont want to go back to chicago i liked not hearing about bad politicians or oprah i hate oprah only 4 days left in the uk|1.0           |0       |
|being ill sucks                                    

+--------------------------------------------------------------------------------------------------------------------------------+--------------+--------+
|cleaned_text                                                                                                                    |toxicity_score|polarity|
+--------------------------------------------------------------------------------------------------------------------------------+--------------+--------+
|tweet4today quothave a limbo party while you are still supple enough to get under that barquot still                            |0.04292183    |0       |
|hang on does anyone use fax machines any more                                                                                   |0.125645      |0       |
|so bored still no internet at home                                                                                              |3.8403085E-20 |0       |
|is home but has to revise                                            

## Feature Engineering

For further analysis, we need to aggregate data by users and by date/time

For users:
- tweet_count – number of user tweets

- avg_toxicity – average probability of toxicity (from 0 to 1)

- toxic_ratio – the proportion of tweets that the model labeled as toxic (at a threshold)

- avg_sentiment_pos_ratio – the proportion of positive tweets the user has (from 0 to 1, where 0 = always negative, 1 = always positive)

For time:
- total_tweets - numof tweets at that day & hour
- avg_toxicity - probability of toxic tweet
- toxic_ratio - part of tweets marked as toxic
- avg_sentiment_pos_ration - numof positive (NOT non-toxic) tweets

In [10]:
# agrregate avg toxicity and toxicity part in users
# make polarity binary
user_features = df_scored.groupBy("user").agg(
    count("*").alias("tweet_count"),
    mean("toxicity_score").alias("avg_toxicity"),
    # part of toxic yweets
    (spark_sum(when(col("is_toxic") == 1, 1).otherwise(0)) / count("*")).alias("toxic_ratio"),
    # mean sentiment (normalized)
    (mean(col("polarity")) / 4).alias("avg_sentiment_pos_ratio")
)

user_features.show(5, truncate=False)
print(f"Number of unique users: {user_features.count()}")

+-------------+-----------+---------------------+-----------+-----------------------+
|user         |tweet_count|avg_toxicity         |toxic_ratio|avg_sentiment_pos_ratio|
+-------------+-----------+---------------------+-----------+-----------------------+
|KatieAngell  |1          |3.799648811986028E-39|0.0        |0.0                    |
|ace587       |4          |0.003635118482635914 |0.0        |0.5                    |
|OhShayLaVie  |2          |0.00924999825656414  |0.0        |0.0                    |
|gildardomunoz|1          |0.0                  |0.0        |0.0                    |
|chann16      |2          |1.83756824867487E-7  |0.0        |0.0                    |
+-------------+-----------+---------------------+-----------+-----------------------+
only showing top 5 rows


Number of unique users: 658713


In [11]:
# aggregate by time and date

time_features = df_scored.groupBy("weekday", "hour").agg(
    count("*").alias("total_tweets"),
    mean("toxicity_score").alias("avg_toxicity"),
    # toxic part: sum(is_toxic) / count(*)
    (spark_sum(col("is_toxic")) / count("*")).alias("toxic_ratio"),
    # avg sentiment (normalised на 0-1)
    (mean("polarity") / 4).alias("avg_sentiment_pos_ratio")
).orderBy("weekday", "hour")

time_features.show(10)

+-------+----+------------+-------------------+-------------------+-----------------------+
|weekday|hour|total_tweets|       avg_toxicity|        toxic_ratio|avg_sentiment_pos_ratio|
+-------+----+------------+-------------------+-------------------+-----------------------+
|      0|   0|       17180|0.08264461487242988|0.07665890570430733|     0.6326542491268917|
|      0|   1|       16633|0.08745851967648655|0.08140443696266458|     0.6537004749594181|
|      0|   2|       16522|0.09079823328248422|0.08485655489650164|     0.6510107735141024|
|      0|   3|       16957|0.08768271059570468| 0.0809105384207112|     0.6311257887598042|
|      0|   4|       17732|0.08139315688708447|0.07551319648093842|     0.6042747575005639|
|      0|   5|       18314| 0.0791462916474306|0.07316806814458884|     0.5900950092825161|
|      0|   6|       18590|0.08313477091843689|0.07681549220010758|       0.57235072619688|
|      0|   7|       18255|0.08137651024085985|0.07625308134757601|     0.570528

## Save

In [12]:
df_scored.select("user", "cleaned_text", "polarity", "weekday", "hour",
                 "toxicity_score", "is_toxic") \
    .write.mode("overwrite").parquet("/kaggle/working/scored_tweets.parquet")

user_features.write.mode("overwrite").parquet("/kaggle/working/user_features.parquet")

time_features.write.mode("overwrite").parquet("/kaggle/working/time_features.parquet")

print("All three datasets saved: scored_tweets, user_features, time_features")

All three datasets saved: scored_tweets, user_features, time_features
